In [0]:
# --- 1. Define All Project Parameters ---
# This block defines all widgets for the entire project,
# providing default values for manual runs or for the main job.

dbutils.widgets.removeAll()

# --- Catalog and Schema Parameters ---
dbutils.widgets.text("catalog", "iowa_sales", "Catalog Name")
dbutils.widgets.text("bronze_schema", "sales_bronze", "Bronze Schema Name")
dbutils.widgets.text("silver_schema", "sales_silver", "Silver Schema Name")
dbutils.widgets.text("gold_schema", "sales_gold", "Gold Schema Name")

# --- Volume Path Parameters ---
dbutils.widgets.text("process_path", "/Volumes/iowa_sales/sales_bronze/process", "Source Process Path")
dbutils.widgets.text("processed_path", "/Volumes/iowa_sales/sales_bronze/processed", "Source Processed Path")

# --- Table Name Parameters ---
dbutils.widgets.text("bronze_table", "sales_raw", "Bronze Table Name")
dbutils.widgets.text("silver_table", "sales_cleaned", "Silver Table Name")  
dbutils.widgets.text("fact_sales", "fact_sales", "Gold Fact Table Name")
dbutils.widgets.text("dim_date", "dim_date", "Date Dimension Table Name")
dbutils.widgets.text("dim_vendor", "dim_vendor", "Vendor Dimension Table Name")
dbutils.widgets.text("dim_store", "dim_store", "Store Dimension Table Name")
dbutils.widgets.text("dim_product", "dim_product", "Product Dimension Table Name")

# --- 2. Get All Project Parameters ---
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

process_path = dbutils.widgets.get("process_path")
processed_path = dbutils.widgets.get("processed_path")

bronze_table = dbutils.widgets.get("bronze_table")
silver_table = dbutils.widgets.get("silver_table")
fact_sales = dbutils.widgets.get("fact_sales")
dim_date = dbutils.widgets.get("dim_date")
dim_vendor = dbutils.widgets.get("dim_vendor")
dim_store = dbutils.widgets.get("dim_store")
dim_product = dbutils.widgets.get("dim_product")

In [0]:
# --- 4. Execute Dynamic SQL Setup ---
# Use spark.sql() with f-strings to inject the Python variables

print(f"Creating catalog '{catalog}' if it does not exist...")
spark.sql(f"""
  CREATE CATALOG IF NOT EXISTS {catalog}
  COMMENT 'Catalog for the Databricks Academy Iowa Liquor Sales project.'
""")

print(f"Creating schema '{catalog}.{bronze_schema}' if it does not exist...")
spark.sql(f"""
  CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}
  COMMENT 'Schema for raw, unprocessed source data.'
""")

print(f"Creating schema '{catalog}.{silver_schema}' if it does not exist...")
spark.sql(f"""
  CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}
  COMMENT 'Schema for validated, cleansed, and conformed data.'
""")

print(f"Creating schema '{catalog}.{gold_schema}' if it does not exist...")
spark.sql(f"""
  CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}
  COMMENT 'Schema for curated, aggregated data for analytics (star schema).'
""")

print(f"Creating volume '{catalog}.{bronze_schema}.process' if it does not exist...")
spark.sql(f"""
  CREATE VOLUME IF NOT EXISTS {catalog}.{bronze_schema}.process
  COMMENT 'Volume for landing raw source files for the project.'
""")

print(f"Creating volume '{catalog}.{bronze_schema}.processed' if it does not exist...")
spark.sql(f"""
  CREATE VOLUME IF NOT EXISTS {catalog}.{bronze_schema}.processed
  COMMENT 'Volume for archiving processed raw files.'
""")

print("--- Initial setup complete. ---")

## **Creating the Date Dimension**

In [0]:
import traceback

try:
    # 1. Define and execute the table creation query
    create_query = f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.{dim_date} (
      date_key INT,
      full_date DATE,
      day INT,
      month INT,
      month_name STRING,
      quarter INT,
      year INT,
      day_of_week INT,
      day_name STRING,
      week_of_year INT,
      is_a_holliday BOOLEAN
    )
    USING DELTA
    COMMENT 'Date Dimension'
    """
    spark.sql(create_query)

    # 2. Define and execute the population query
    populate_query = f"""
    INSERT OVERWRITE {catalog}.{gold_schema}.{dim_date}
    WITH date_sequence AS (
      SELECT explode(sequence(TO_DATE('2012-01-01'), TO_DATE('2025-12-31'), INTERVAL 1 DAY)) AS full_date
    )
    SELECT
      YEAR(full_date) * 10000 + MONTH(full_date) * 100 + DAY(full_date) AS date_key,
      full_date,
      DAY(full_date) AS day,
      MONTH(full_date) AS month,
      DATE_FORMAT(full_date, 'MMMM') AS month_name,
      QUARTER(full_date) AS quarter,
      YEAR(full_date) AS year,
      DAYOFWEEK(full_date) AS day_of_week,
      DATE_FORMAT(full_date, 'EEEE') AS day_name,
      WEEKOFYEAR(full_date) AS week_of_year,
      FALSE AS is_a_holliday
    FROM date_sequence
    """
    spark.sql(populate_query)
    
    print(f"Date dimension {catalog}.{gold_schema}.{dim_date} is ready.")

except Exception as e:
    print(f"Failed to create or populate date dimension: {str(e)}")
    print(traceback.format_exc())
    raise e